# Assignment APIs tutorial

In this notebook we are using the python package "millionaire-client" to interact with the deployed application for the NLP assignment 2026.

Required files:
- Directory called "millionaire_client"
- 1 colab notebook

Both files must be saved in a directory in your Google Drive, for example:
```
gDrive_home/
├── Colab Notebooks/
│   └── NLP_assignment/
│       ├── PoliMillionaire.ipynb <-- Your notebook
│       └── millionaire_client/ <-- Directory provided
```

### Sign up procedure
Before showing you how the api work, you need to signup from a web browser.
- Paste this link into your browser [http://131.175.15.22:51111/](http://131.175.15.22:51111/) this is where the demo is deployed
- You will see a standard login/sign up screen, please click on sign up
- In the "email" field please enter your politecnico email, you are allwed to create only 1 account using the same email you registered to the NLP course
- Choose whever username/password you prefer (be creative ;))

### Game interaction

Once you signed up, you can start interacting already from the api.

First of all, let's connect your drive to this Colab Notebook

In [ ]:
from google.colab import drive
import os
drive.mount('/content/gdrive/')

Then we need to add our python package "millionaire_client" to the system path, so python can see it.

In [ ]:
import sys
import os

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment_api_client'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

Let's import the client classes

In [ ]:
from millionaire_client import MillionaireClient, AuthenticationError

You can save your password in a Colab secret (the "key" icon on the tab on the left) and import it into your notebook.

In [ ]:
from google.colab import userdata
pwd = userdata.get('poli-millionaire')

Now keep the API_URL as stated, but please change the username and password to be the ones you used during sign up session.

In [ ]:
API_URL = "http://131.175.15.22:51111/"
username = "nicolo"
password = pwd

Now we can instantiate a MillionaireClient object and call the login method, which takes as parameters username and password.

In [ ]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

After login, the web page is showing you different types of competitions, for each of them you can choose to play a game or to see the leaderboard. For now let's list all of the.

In [ ]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")

In [ ]:
# Choose a competition ID
comp_id = 1

After choosing a competition, we can start a game! We can choose to start a game by calling `game = client.game.start(competition_id=comp_id)`. The object game is the one that is handling the game itself, we can call:
- game.current_question.text : to know the current question in text format
- game.current_level: to check the current level of difficulty of the question
- game.current_question.options: to check the possible choices we have to answer the question
- game.answer: to send to the server the answer we choose (the integer corresponding to our choice) and get the response (either correct or incorrect)

WATCH OUT! Each question has a timer, you have maximum 30 seconds to answer the question. As of now, if you exceed the maximum allowed time, there is not a "push notification". You still have to submit your answer anyway and, even though the answer was correct, you will get a TimedOut response!

In [ ]:
def play_game(game):
  # Play the game
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      # Get time remaining
      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      # Get answer
      try:
          answer_input = input("\nYour answer (option ID): ").strip()
          answer_id = int(answer_input)
      except ValueError:
          print("Invalid input. Please enter a number.")
          continue

      # Submit answer
      result = game.answer(answer_id)

      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [ ]:
# Start the game
print("\n=== Starting Game ===")
game = client.game.start(competition_id=comp_id)
print(f"Session ID: {game.session_id}")
print(f"Total number of questions: {game.state.competition.max_levels}")
print()
play_game(game)

In [ ]:
# Show leaderboard position
lb = client.leaderboard.get(competition_id=comp_id, limit=10)
print(f"\n=== Leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:5], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")

 ## Speech interaction

In [ ]:
from IPython.display import Audio, display

In [ ]:
def save_audio(data: bytes, filename: str):
    """Save audio bytes to a WAV file."""
    with open(filename, "wb") as f:
        f.write(data)
    print(f"  Saved: {filename}")

In [ ]:
def play_game(game):
    while game.in_progress:
        print(f"\n--- Level {game.current_level} ---")

        # Fetch question audio
        print("Fetching question audio...")
        try:
            question_audio = game.fetch_audio_question()
            save_audio(question_audio, f"level_{game.current_level}_question.wav")
            display(Audio(question_audio))
        except GameError as e:
            print(f"Error fetching question audio: {e}")
            break

        # Fetch option audios sequentially (A, B, C, D)
        option_map = {}
        audios = []
        for i in range(4):
            letter = chr(65 + i)  # A, B, C, D
            print(f"Fetching option {letter} audio...")
            try:
                option_audio = game.fetch_audio_option_next()
                save_audio(option_audio, f"level_{game.current_level}_option_{letter}.wav")
                audios.append(option_audio)
            except GameError as e:
                print(f"Error fetching option audio: {e}")
                break

            # Store mapping for answer submission
            if game.current_question and i < len(game.current_question.options):
                option_map[letter] = game.current_question.options[i].id

        display(Audio(audios[0]),
                Audio(audios[1]),
                Audio(audios[2]),
                Audio(audios[3])) # Pardon this horrific hard-coded stuff, due to (even more horrific) ipykernel logic. ~Federico

        if not option_map:
            print("No question available. Game may have ended.")
            break

        game.refresh_state()

        # Get time remaining (timer starts after Option D is fetched)
        time_left = game.time_remaining
        if time_left is not None:
            print(f"\nTime remaining: {time_left:.1f}s")

        # Get answer
        answer_input = input("\nYour answer (A/B/C/D): ").strip().upper()
        if answer_input not in option_map:
            print("Invalid input. Please enter A, B, C, or D.")
            continue

        answer_id = option_map[answer_input]
        result = game.answer(answer_id)

        # Handle timeout
        if result.timed_out or result.status == "timeout":
            print(" TIME'S UP!")
            print(f"\n Game Over! You ran out of time.")
            print(f" Final earnings: ${result.earned_amount:,.2f}")
            break

        if result.correct:
            print(" CORRECT!")
            if result.game_over:
                print(f"\n CONGRATULATIONS! You completed the game!")
                print(f" Final earnings: ${result.earned_amount:,.2f}")
            else:
                print(f" Earned so far: ${result.earned_amount:,.2f}")
        else:
            print(" WRONG!")
            print(f"\n Game Over!")
            print(f" Final earnings: ${result.earned_amount:,.2f}")
    print("\n=== Game Summary ===")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [ ]:
# Start speech mode game
print("\n=== Starting Game (With mode selection)===")
game = client.game.start(competition_id=comp_id, mode="speech")
print(f"Session ID: {game.session_id}")
print(f"Mode: {game.mode}")
print()
play_game(game)

In speech mode, the 30 seconds timer is starting when we are requesting the last option!!

In [ ]:
# Show leaderboard position (speech mode)
lb = client.leaderboard.get(competition_id=comp_id, limit=10, mode="speech") # <---- include mode!!
print(f"\n=== Speech Leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:5], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")